# Thực nghiệm Naive Bayes đếm tần suất

Bài toán: dự đoán Potability (0/1) từ chín chỉ tiêu. Mọi thuật toán nằm trong mã nguồn tự cài. Notebook đọc lần chạy đã lưu, không fit lại trên tập kiểm tra.


In [ ]:
import json
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

from src.config import DEMO_RUN_ID, FEATURE_COLUMNS
from src.algorithms.naive_bayes import NaiveBayes
from src.pipeline import load_pipeline, sha256_file

run = json.loads((root / "results" / "runs" / f"{DEMO_RUN_ID}.json").read_text(encoding="utf-8"))
assert run["stage"] == "evaluated", "Cần chạy train và evaluate theo README."
for name, checksum in run["files"].items():
    if sha256_file(root / name) != checksum:
        raise ValueError(f"Checksum không khớp: {name}")
pipeline = load_pipeline(root / run["model_path"], run["model_sha256"])
assert pipeline.run_id == DEMO_RUN_ID
run["configuration"]

## Ví dụ tính tay

Ví dụ tổng hợp với hai thuộc tính. Cho đầu vào (1, 0): điểm lớp 0 là (3/5)(1/3)(2/3) = 2/15; điểm lớp 1 là (2/5)(2/2)(1/2) = 1/5. Do đó hậu nghiệm lần lượt bằng 0,4 và 0,6.

In [ ]:
example = NaiveBayes().fit([[0, 0], [0, 1], [1, 0], [1, 1], [1, 0]], [0, 0, 0, 1, 1], [2, 2])
example.predict_with_proba([[1, 0]])

## Ngưỡng đã học và kết quả

Ngưỡng được học từ development; trong CV mỗi fold học ngưỡng và trung vị của riêng phần train. Chỉ số test bên dưới lấy từ cùng run đã được kiểm tra checksum.

In [ ]:
from IPython.display import Markdown, display

state = pipeline.preprocessor.to_state()
rows = ["| Đặc trưng | Ngưỡng chia | Trung vị |", "| --- | --- | --- |"]
for column, name in enumerate(FEATURE_COLUMNS):
    cuts = ", ".join(f"{cut:.4f}" for cut in state["cut_points"][column])
    rows.append(f"| {name} | {cuts} | {state['fill_values'][column]:.4f} |")
display(Markdown("\n".join(rows)))

In [ ]:
metrics_dir = root / "results" / "metrics" / DEMO_RUN_ID
cv = json.loads((metrics_dir / "cv_results.json").read_text(encoding="utf-8"))
print("CV:", {name: value["summary"] for name, value in cv.items()})
fields = ("accuracy", "balanced_accuracy", "macro_f1", "roc_auc", "confusion_matrix")
for name in ("test_metrics.json", "baseline_test_metrics.json"):
    metrics = json.loads((metrics_dir / name).read_text(encoding="utf-8"))
    print(name, {field: metrics[field] for field in fields})

Điểm số chỉ phục vụ đối chiếu trên bộ dữ liệu này. Đầu ra không thay thế kiểm nghiệm chất lượng nước.